In [1]:
data=[
    {"tag":"greeting",
     "patterns":["hello","hi","goodmorning","good evening"],
     "responses":["Hello!","Hi,how can I help you?" ]},
    {"tag":"goodbye",
     "patterns":["bye","see you","goodbye"],
     "responses":["Goodbye!","see you soon!"]},
    {"tag":"password_reset",
     "patterns":["reset password","forgot password","change my password"],
     "responses":["You can reset your password by clicking 'Forgot Password' on the login screen."]},
    {"tag":"contact",
     "patterns":["How can I contact support?", "I need help", "Customer service number"],
     "responses":["You can reach support@example.com or call 123-456-7890."]},
    {"tag":"weather",
     "patterns":["Whats the weather like?","Is it going to rain today?","Tell me the weather"],
     "responses":["I'm not connected to live weather data, but you can check weather.com for accurate info."]},
    {"tag":"Language Translation",
     "patterns":["How do you say 'hello' in spanish?"],
     "responses":["'hello in spanish is 'Hola'"]}
]

In [2]:
import json
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
import pickle
import os

In [3]:
with open("intents.json",'w') as outfile:
  json.dump(data,outfile,indent=4)


training_sentences =[]
training_labels = []
labels =[]
responses ={}

for intent in data:
  for pattern in intent["patterns"]:
    training_sentences.append(pattern)
    training_labels.append(intent["tag"])

  responses[intent["tag"]] = intent["responses"]


  if intent["tag"] not in labels:
    labels.append(intent["tag"])

lbl_encoder=LabelEncoder()
lbl_encoder.fit(training_labels)
training_labels_encoded = lbl_encoder.transform(training_labels)


vocab_size=1000
embedding_dim=16
max_len = 20
oov_token ="<oov>"

tokenizer =Tokenizer(num_words=vocab_size,oov_token=oov_token)
tokenizer.fit_on_texts(training_sentences)
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(training_sentences)
padded_sequences = pad_sequences(sequences, truncating = 'post', maxlen=max_len)

model = Sequential([
    Embedding(vocab_size,embedding_dim,input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(16, activation ='relu'),
    Dense(16, activation='relu'),
    Dense(len(labels),activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

if not os.path.exists("chat_model.h5"):
  model.fit(padded_sequences, np.array(training_labels_encoded), epochs=500)
  model.save("chat_model.h5")
  pickle.dump(tokenizer, open('tokenizer.pkl','wb'))
  pickle.dump(lbl_encoder, open('label_encoder.pkl','wb'))
else:
  model=tf.keras.models.load_model("chat_model.h5")
  tokenizer = pickle.load(open('tokenizer.pkl','rb'))
  lbl_encoder = pickle.load(open('label_encoder.pkl','rb'))

print("chatbot is ready! Type 'quit' to exit.")
while True:
  user_input = input("you: ")
  if user_input.lower() == "quit":
    print("Bot: GoodBye!")
    break

  input_seq = tokenizer.texts_to_sequences([user_input])
  padded=pad_sequences(input_seq,truncating='post',maxlen=max_len)
  prediction = model.predict(padded, verbose=0)
  tag = lbl_encoder.inverse_transform([np.argmax(prediction)])

  print("Bot:", random.choice(responses[tag[0]]))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.1765 - loss: 1.7897
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.1765 - loss: 1.7880
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.2941 - loss: 1.7863
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.3529 - loss: 1.7846
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.2353 - loss: 1.7831
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.2353 - loss: 1.7817
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.2353 - loss: 1.7802
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.2353 - loss: 1.7786
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.2353 - loss: 1.7770
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.2353 - loss: 1.7754
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.2353 - loss: 1.7738
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.2353 - loss

chatbot is ready! Type 'quit' to exit.
you: hi
Bot: Hi,how can I help you?
you: how do you say hello in spanish
Bot: You can reset your password by clicking 'Forgot Password' on the login screen.
you: quit
Bot: GoodBye!
